# Playbook: Kaggle Tour

> **Курс «От нуля до своих агентов» — Модуль 1.5.**
> Это не урок-теория, это **демо-сценарий** для живой практики.
> Веду демонстрацию в Kaggle UI и время от времени запускаю ячейки этой
> ноутбука — чтобы все вместе видели, что получилось.
>
> Длительность: ~25 минут.
> Что в конце: у каждого ученика — свой работающий Kaggle-ноутбук с
> подключённым GPU и привязанным секретом `GOOGLE_API_KEY`.

## 0. Что мы делаем сегодня

Три задачи на эту демонстрацию:

1. Завести аккаунт на Kaggle и пройти phone verify (без него нет GPU и
   нет Secrets — это два самых ценных бесплатных ресурса).
2. Создать первый ноутбук, переключить его на GPU T4 и убедиться, что
   GPU действительно подключён.
3. Положить API-ключ Google AI Studio в Kaggle Secrets, достать его из
   кода и сделать первый запрос к Gemini.

Это база, на которой будут стоять все следующие ДЗ курса.

## 1. Регистрация и phone verify (web, не код)

1. Открыть https://www.kaggle.com → **Register**.
2. Регистрация через Google-аккаунт быстрее всего и сразу даёт тот
   аккаунт, который пригодится для AI Studio.
3. Подтвердить почту по ссылке из письма.
4. Зайти в https://www.kaggle.com/settings → секция **Phone verification**.
5. Указать номер, ввести код из SMS.

**Почему phone verify — обязательный шаг:**
без него Kaggle не показывает кнопку Accelerator (GPU/TPU) в ноутбуке
и не даёт работать с Secrets. Это анти-абьюз мера, обойти нельзя.

**Если номер не из поддерживаемого региона:** заведите аккаунт на
номер, к которому есть доступ (виртуальный SIM, друг, рабочий номер).

## 2. Создаём первый ноутбук (web)

1. Слева в боковом меню: **Code** → **+ New Notebook**.
2. Откроется Jupyter в браузере. По умолчанию даётся CPU.
3. Справа есть панель **Notebook options**. Главные настройки там:
   - **Accelerator** — `None` / `GPU T4 x2` / `GPU P100` / `TPU VM`.
   - **Persistence** — `Files only` достаточно (переменные между
     запусками всё равно теряются, оставляем по умолчанию).
   - **Language** — Python.

Сейчас оставим Accelerator = `None` — для следующей ячейки GPU не
нужен. Включим его на шаге 3, чтобы заодно посмотреть, как это
выглядит.

### 2.1 Sanity check — что мы вообще запускаем

Первая ячейка любого нового ноутбука у меня — это «привет, какая среда».
Это занимает 3 секунды и сразу видно: правильная версия Python, какая
ОС, есть ли GPU.

In [ ]:
import sys
import platform

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

## 3. Включаем GPU и проверяем

Справа в **Notebook options** → **Accelerator** → выбираем
**GPU T4 x2**. Ноутбук перезапустится (ядро будет новое).

После перезапуска проверим, что GPU действительно подключён, через
`nvidia-smi`. Если в выводе видна строка `Tesla T4` — всё хорошо.

In [ ]:
!nvidia-smi 2>/dev/null | head -20 || echo "GPU не подключён — переключите Accelerator справа"

**Что важно запомнить:**

- GPU-квота — около 30 часов в неделю на верифицированный аккаунт.
  Сбрасывается каждую субботу.
- Когда ноутбук «активный» (вкладка открыта или код выполняется) —
  квота тикает. Когда вы закрыли вкладку и нажали `Stop session`
  внизу — не тикает.
- На учебные ДЗ 30 часов хватает с большим запасом. Сжигают её обычно
  «забыл выключить, ушёл спать, проснулся — кончилось».

## 4. Secrets — главный приём всего курса

Сейчас нам нужен API-ключ Google AI Studio. Параллельно у меня в
соседней вкладке открыт https://aistudio.google.com → **Get API key**.
Создаю ключ, копирую (он начинается с `AIza...`).

Дальше — **не вставляю ключ в код**. Вставляю в Kaggle Secrets:

1. В верхнем меню ноутбука: **Add-ons** → **Secrets**.
2. **Add a new secret**.
3. Label: `GOOGLE_API_KEY` (имя в верхнем регистре, без пробелов).
4. Value: сам ключ.
5. **Обязательно ставим галочку «Attach to notebook»** — иначе ноутбук
   секрет не увидит.

В коде секрет достаётся вот так:

In [ ]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
print("Ключ загружен, длина:", len(GOOGLE_API_KEY), "символов")
# Сам ключ НИКОГДА не печатаем. Только длину — для проверки, что он
# не пустой.

**Самая частая ошибка:** `No user secrets exist for kernel id ...`
Лечится одной галкой «Attach to notebook» рядом с именем секрета.
В 90% случаев студент создаёт секрет, но забывает его прикрепить.

## 5. Первый запрос к Gemini — проверяем end-to-end

Теперь, когда у нас есть ключ, делаем первый осмысленный запрос —
просто чтобы убедиться, что вся цепочка (Kaggle → Secrets → Gemini API
→ ответ) работает.

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Поздоровайся со студентами курса \"От нуля до своих агентов\" одним предложением.",
)

print(response.text)

Если ответ пришёл — поздравляем, всё работает.

Если упало с `User location is not supported` — это региональное
ограничение Gemini API. Лечится: (а) VPN, (б) выпустить ключ из самого
Kaggle-ноутбука (Google видит запрос с серверов GCP).

## 6. Datasets — подключаем готовый датасет за один клик

Kaggle сам по себе — это огромная коллекция датасетов. Можно
подключить любой к своему ноутбуку без скачивания.

1. Справа в панели — **+ Add Input**.
2. **Datasets** → поиск по названию (например, `imdb dataset`).
3. Нажимаем на иконку плюса рядом с нужным датасетом.

После подключения файлы появляются в `/kaggle/input/<dataset-slug>/`.
Покажем:

In [ ]:
import os

# Эту ячейку имеет смысл запускать ПОСЛЕ того, как вы подключили
# какой-нибудь датасет через UI. Если ничего не подключали — список
# будет пустым.
input_root = "/kaggle/input"
if os.path.isdir(input_root):
    for name in sorted(os.listdir(input_root)):
        print("-", name)
else:
    print("Нет подключённых датасетов. Добавьте через '+ Add Input'.")

## 7. Сохранение и публикация

Когда ДЗ готово:

1. Сверху: **Save Version** → **Save & Run All (Commit)** — это полный
   прогон ноутбука сверху вниз с нуля. Кэш ячеек сбрасывается. Долго,
   но именно так Kaggle сохраняет финальную версию.
2. **Share** → **Public** — теперь ссылку видно всем.
3. Скопируйте URL вида `https://www.kaggle.com/code/<your-id>/<slug>` —
   это и есть артефакт для сдачи.

**Перед публикацией обязательно:**

- Убрать из вывода ячеек всё, что может содержать ваш ключ.
- Если случайно напечатали — удалите ячейку, нажмите `Save & Run All`
  ещё раз, и **сразу отзовите ключ** в AI Studio. Раскрытый ключ —
  бесплатный билет в чужие эксперименты на ваш счёт.

## 8. Что должно получиться у каждого

К концу демонстрации у каждого ученика должно быть:

- [ ] Аккаунт Kaggle с пройденным phone verify.
- [ ] Хотя бы один ноутбук, в котором запускался `!nvidia-smi` с GPU.
- [ ] Секрет `GOOGLE_API_KEY` создан и прикреплён к ноутбуку.
- [ ] Ячейка `client.models.generate_content(...)` вернула осмысленный
      ответ.

Если хоть один пункт не получился — это блокер на ближайшие модули.
Останавливаемся и разбираем индивидуально, остальная группа делает
паузу.

**Следующий шаг — playbook по Hugging Face.**